## 1. 패키지 설치 및 Import

In [1]:
import os
import yaml
from pathlib import Path
from typing import Literal, List, Optional
from pydantic import BaseModel, Field
from dspydantic import PydanticOptimizer, Example, create_optimized_model
import dspy

# 프로젝트 루트 설정
PROJECT_ROOT = Path.cwd().parent
PROMPTS_DIR = PROJECT_ROOT / "generate_synthetic_table" / "prompts"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Prompts Directory: {PROMPTS_DIR}")

Project Root: /Users/jaehyeokchoi/Desktop/TableMagnifier
Prompts Directory: /Users/jaehyeokchoi/Desktop/TableMagnifier/generate_synthetic_table/prompts


## 2. 기존 프롬프트 로드

In [2]:
def load_prompts(yaml_file: str = "default.yaml") -> dict:
    """YAML 파일에서 프롬프트 로드"""
    path = PROMPTS_DIR / yaml_file
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

# 기존 프롬프트 로드
prompts = load_prompts()
print("로드된 프롬프트:")
for key in prompts.keys():
    print(f"  - {key}")

로드된 프롬프트:
  - generate_qa
  - generate_qa_from_image
  - generate_synthetic_table
  - generate_synthetic_table_from_image
  - image_to_html
  - parse_contents
  - parse_synthetic_table
  - revise_synthetic_table
  - revise_synthetic_table_from_image
  - self_reflection
  - self_reflection_from_image
  - validate_parsed_table


## 3. Pydantic 모델 정의

각 프롬프트의 출력 형식에 맞는 Pydantic 모델을 정의합니다.

In [3]:
# ========================================
# QA Generation 모델
# ========================================
class QAPair(BaseModel):
    """단일 QA 쌍"""
    question: str = Field(description="테이블 데이터에 기반한 질문 (한국어)")
    answer: str = Field(description="질문에 대한 정확한 답변")
    type: Literal["lookup", "comparison", "calculation", "reasoning"] = Field(
        description="질문 유형: lookup(조회), comparison(비교), calculation(계산), reasoning(추론)"
    )

class QAGenerationOutput(BaseModel):
    """QA 생성 프롬프트의 출력 형식"""
    qa_pairs: List[QAPair] = Field(
        description="테이블에서 생성된 다양한 유형의 QA 쌍 목록"
    )

# ========================================
# Self-Reflection 모델
# ========================================
class ReflectionIssue(BaseModel):
    """발견된 이슈"""
    type: Literal["structure", "header", "row_count", "data_type", "html_validity", "other"] = Field(
        description="이슈 유형"
    )
    detail: str = Field(description="이슈에 대한 상세 설명")

class SelfReflectionOutput(BaseModel):
    """Self-Reflection 프롬프트의 출력 형식"""
    passed: bool = Field(description="검증 통과 여부")
    score: int = Field(description="품질 점수 (0-100)")
    issues: List[ReflectionIssue] = Field(
        description="발견된 문제점 목록"
    )
    revision_instructions: str = Field(
        description="테이블 수정을 위한 구체적인 단계별 지시사항"
    )

# ========================================
# Table Validation 모델
# ========================================
class TableValidationOutput(BaseModel):
    """테이블 검증 프롬프트의 출력 형식"""
    valid: bool = Field(description="테이블 유효성 여부")
    reason: str = Field(description="판단 이유에 대한 간단한 설명")

print("✅ Pydantic 모델 정의 완료")

✅ Pydantic 모델 정의 완료


## 4. 학습 예제 정의

최적화에 사용할 예제들을 정의합니다. 실제 테이블 데이터와 기대 출력을 제공합니다.

In [4]:
# ========================================
# QA Generation 예제
# ========================================

qa_examples = [
    Example(
        text="""<table>
        <tr><th>구분</th><th>2023년</th><th>2024년</th></tr>
        <tr><td>매출액</td><td>1,000억</td><td>1,200억</td></tr>
        <tr><td>영업이익</td><td>100억</td><td>150억</td></tr>
        <tr><td>순이익</td><td>80억</td><td>120억</td></tr>
        </table>""",
        expected_output=QAGenerationOutput(
            qa_pairs=[
                QAPair(question="2024년 매출액은 얼마입니까?", answer="1,200억", type="lookup"),
                QAPair(question="2023년과 2024년 중 영업이익이 더 높은 연도는?", answer="2024년", type="comparison"),
                QAPair(question="2023년 대비 2024년 순이익 증가액은?", answer="40억 (120억 - 80억)", type="calculation"),
                QAPair(question="매출 대비 영업이익률의 변화 추세는 어떠합니까?", answer="2023년 10%에서 2024년 12.5%로 개선됨", type="reasoning"),
            ]
        )
    ),
    Example(
        text="""<table>
        <tr><th>학생</th><th>국어</th><th>영어</th><th>수학</th></tr>
        <tr><td>김철수</td><td>85</td><td>90</td><td>95</td></tr>
        <tr><td>이영희</td><td>90</td><td>85</td><td>88</td></tr>
        <tr><td>박민수</td><td>78</td><td>92</td><td>80</td></tr>
        </table>""",
        expected_output=QAGenerationOutput(
            qa_pairs=[
                QAPair(question="김철수의 수학 점수는?", answer="95", type="lookup"),
                QAPair(question="영어 점수가 가장 높은 학생은?", answer="박민수 (92점)", type="comparison"),
                QAPair(question="이영희의 총점은?", answer="263점 (90+85+88)", type="calculation"),
                QAPair(question="세 학생 중 전과목 점수가 가장 고른 학생은?", answer="이영희 (점수 편차가 가장 작음)", type="reasoning"),
            ]
        )
    ),
    Example(
        text="""<table>
        <tr><th>제품</th><th>가격</th><th>재고</th><th>판매량</th></tr>
        <tr><td>노트북</td><td>1,500,000원</td><td>50</td><td>120</td></tr>
        <tr><td>마우스</td><td>35,000원</td><td>200</td><td>450</td></tr>
        <tr><td>키보드</td><td>89,000원</td><td>150</td><td>280</td></tr>
        </table>""",
        expected_output=QAGenerationOutput(
            qa_pairs=[
                QAPair(question="노트북의 가격은 얼마입니까?", answer="1,500,000원", type="lookup"),
                QAPair(question="판매량이 가장 많은 제품은?", answer="마우스 (450개)", type="comparison"),
                QAPair(question="마우스와 키보드의 가격 차이는?", answer="54,000원 (89,000 - 35,000)", type="calculation"),
                QAPair(question="재고 대비 판매량 비율이 가장 높은 제품은?", answer="노트북 (재고 50개 대비 판매 120개로 240%)", type="reasoning"),
            ]
        )
    ),
]

print(f"✅ QA Generation 예제: {len(qa_examples)}개")

✅ QA Generation 예제: 3개


In [5]:
# ========================================
# Self-Reflection 예제 (최소 3개 이상 필요)
# ========================================

reflection_examples = [
    Example(
        text={
            "original_html": """<table>
            <tr><th>이름</th><th>부서</th><th>연봉</th></tr>
            <tr><td>김철수</td><td>개발팀</td><td>5,000만원</td></tr>
            <tr><td>이영희</td><td>마케팅</td><td>4,500만원</td></tr>
            </table>""",
            "synthetic_html": """<table>
            <tr><th>이름</th><th>부서</th><th>연봉</th></tr>
            <tr><td>박민수</td><td>인사팀</td><td>4,800만원</td></tr>
            <tr><td>최지은</td><td>기획팀</td><td>5,200만원</td></tr>
            </table>"""
        },
        expected_output=SelfReflectionOutput(
            passed=True,
            score=95,
            issues=[],
            revision_instructions="No revisions needed. Structure and data types match perfectly."
        )
    ),
    Example(
        text={
            "original_html": """<table>
            <tr><th>제품</th><th>수량</th><th>가격</th></tr>
            <tr><td>A</td><td>100</td><td>10,000원</td></tr>
            <tr><td>B</td><td>50</td><td>20,000원</td></tr>
            </table>""",
            "synthetic_html": """<table>
            <tr><th>제품</th><th>수량</th></tr>
            <tr><td>C</td><td>75</td></tr>
            <tr><td>D</td><td>120</td></tr>
            </table>"""
        },
        expected_output=SelfReflectionOutput(
            passed=False,
            score=40,
            issues=[
                ReflectionIssue(type="structure", detail="원본은 3열이지만 합성 테이블은 2열입니다."),
                ReflectionIssue(type="header", detail="'가격' 열이 누락되었습니다.")
            ],
            revision_instructions="1. '가격' 열을 추가하세요. 2. 각 행에 가격 데이터를 포함하세요."
        )
    ),
    Example(
        text={
            "original_html": """<table>
            <tr><th>날짜</th><th>매출</th><th>비용</th><th>이익</th></tr>
            <tr><td>2024-01</td><td>500만원</td><td>300만원</td><td>200만원</td></tr>
            <tr><td>2024-02</td><td>600만원</td><td>350만원</td><td>250만원</td></tr>
            <tr><td>2024-03</td><td>700만원</td><td>400만원</td><td>300만원</td></tr>
            </table>""",
            "synthetic_html": """<table>
            <tr><th>날짜</th><th>매출</th><th>비용</th><th>이익</th></tr>
            <tr><td>2024-04</td><td>550만원</td><td>320만원</td><td>230만원</td></tr>
            <tr><td>2024-05</td><td>650만원</td><td>380만원</td><td>270만원</td></tr>
            <tr><td>2024-06</td><td>750만원</td><td>420만원</td><td>330만원</td></tr>
            </table>"""
        },
        expected_output=SelfReflectionOutput(
            passed=True,
            score=98,
            issues=[],
            revision_instructions="No revisions needed. Structure, row count, and data types are consistent."
        )
    ),
    Example(
        text={
            "original_html": """<table>
            <tr><th>학생</th><th>국어</th><th>영어</th><th>수학</th></tr>
            <tr><td>홍길동</td><td>85</td><td>90</td><td>78</td></tr>
            <tr><td>김영희</td><td>92</td><td>88</td><td>95</td></tr>
            </table>""",
            "synthetic_html": """<table>
            <tr><th>학생</th><th>국어</th><th>영어</th><th>수학</th></tr>
            <tr><td>이철수</td><td>팔십오</td><td>구십</td><td>칠십팔</td></tr>
            <tr><td>박민지</td><td>구십이</td><td>팔십팔</td><td>구십오</td></tr>
            </table>"""
        },
        expected_output=SelfReflectionOutput(
            passed=False,
            score=50,
            issues=[
                ReflectionIssue(type="data_type", detail="원본은 숫자 형식(85, 90)이지만 합성은 한글 숫자(팔십오, 구십)입니다.")
            ],
            revision_instructions="1. 점수 열의 값을 아라비아 숫자 형식으로 변경하세요. 예: '팔십오' → '85'"
        )
    ),
    Example(
        text={
            "original_html": """<table>
            <tr><th>도시</th><th>인구</th><th>면적</th></tr>
            <tr><td>서울</td><td>9,776,000</td><td>605.2</td></tr>
            <tr><td>부산</td><td>3,429,000</td><td>769.9</td></tr>
            <tr><td>인천</td><td>2,957,000</td><td>1,063.3</td></tr>
            </table>""",
            "synthetic_html": """<table>
            <tr><th>도시</th><th>인구</th><th>면적</th></tr>
            <tr><td>대구</td><td>2,438,000</td><td>883.5</td></tr>
            <tr><td>광주</td><td>1,456,000</td><td>501.1</td></tr>
            </table>"""
        },
        expected_output=SelfReflectionOutput(
            passed=False,
            score=60,
            issues=[
                ReflectionIssue(type="row_count", detail="원본은 3개 행이지만 합성은 2개 행입니다.")
            ],
            revision_instructions="1. 세 번째 도시 데이터 행을 추가하세요."
        )
    ),
]

print(f"✅ Self-Reflection 예제: {len(reflection_examples)}개")

✅ Self-Reflection 예제: 5개


In [6]:
# ========================================
# Table Validation 예제 (최소 3개 이상 필요)
# ========================================

validation_examples = [
    Example(
        text="""<table>
        <tr><th>항목</th><th>값</th></tr>
        <tr><td>이름</td><td>홍길동</td></tr>
        <tr><td>나이</td><td>30</td></tr>
        </table>""",
        expected_output=TableValidationOutput(
            valid=True,
            reason="올바른 HTML 테이블 구조이며 유의미한 데이터를 포함합니다."
        )
    ),
    Example(
        text="""<table>
        <tr><td></td><td></td></tr>
        <tr><td></td><td></td></tr>
        </table>""",
        expected_output=TableValidationOutput(
            valid=False,
            reason="테이블에 실제 데이터가 없고 모든 셀이 비어있습니다."
        )
    ),
    Example(
        text="<div>This is not a table</div>",
        expected_output=TableValidationOutput(
            valid=False,
            reason="HTML 테이블 태그(<table>)가 없습니다."
        )
    ),
    Example(
        text="""<table>
        <tr><th>제품명</th><th>가격</th><th>재고</th></tr>
        <tr><td>노트북</td><td>1,500,000원</td><td>25</td></tr>
        <tr><td>모니터</td><td>350,000원</td><td>40</td></tr>
        <tr><td>키보드</td><td>89,000원</td><td>100</td></tr>
        </table>""",
        expected_output=TableValidationOutput(
            valid=True,
            reason="올바른 HTML 테이블 구조이며 헤더와 3개의 데이터 행을 포함합니다."
        )
    ),
    Example(
        text="""<table>
        <tr><th>이름</th></tr>
        <tr><td>김철수</td><td>추가데이터</td></tr>
        </table>""",
        expected_output=TableValidationOutput(
            valid=False,
            reason="헤더 열 수(1)와 데이터 열 수(2)가 일치하지 않습니다."
        )
    ),
]

print(f"✅ Table Validation 예제: {len(validation_examples)}개")

✅ Table Validation 예제: 5개


## 5. 프롬프트 최적화 실행

In [7]:
# API 키 설정 - polling_gemini 사용
import sys
import copy
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# ========================================
# polling_gemini를 dspy.LM으로 래핑
# ========================================
from polling_gemini.api_pool import get_gemini_pool

# Gemini API Pool 초기화
gemini_pool = get_gemini_pool()
current_key_info = gemini_pool.api_keys[gemini_pool.current_key_index]

print(f"✅ Gemini API Pool 초기화 완료")
print(f"   - 사용 가능한 API 키: {len(gemini_pool.api_keys)}개")
print(f"   - 현재 키: {current_key_info.name}")
print(f"   - 모델: {gemini_pool.current_model.model_name if gemini_pool.current_model else 'N/A'}")

# dspy.LM으로 Gemini Pool 래핑
class GeminiPoolLM(dspy.LM):
    """polling_gemini를 dspy.LM으로 래핑"""
    
    def __init__(self, pool, model_id: str = "gemini-2.0-flash"):
        super().__init__(model=model_id)
        self.pool = pool
        self.model_id = model_id
    
    def __deepcopy__(self, memo):
        """deepcopy 시 gRPC 채널 복사 문제 방지 - 동일 인스턴스 반환"""
        # gRPC 채널은 복사할 수 없으므로, 새 인스턴스 생성하지만 pool은 공유
        new_instance = GeminiPoolLM.__new__(GeminiPoolLM)
        new_instance.model = self.model
        new_instance.pool = self.pool  # pool은 공유 (deepcopy하지 않음)
        new_instance.model_id = self.model_id
        new_instance.history = []
        new_instance.callbacks = []
        new_instance.num_retries = getattr(self, 'num_retries', 3)
        return new_instance
    
    def copy(self, **kwargs):
        """dspy가 호출하는 copy 메서드 - 동일 pool 공유"""
        new_instance = GeminiPoolLM(self.pool, self.model_id)
        # kwargs 적용 (temperature 등)
        for key, value in kwargs.items():
            if hasattr(new_instance, key):
                setattr(new_instance, key, value)
        return new_instance
        
    def __call__(self, prompt: str = None, messages: list = None, **kwargs) -> list[dict]:
        """dspy.LM 인터페이스 구현 - dspy가 기대하는 형식으로 반환"""
        # messages가 있으면 프롬프트로 변환
        if messages:
            prompt_parts = []
            for msg in messages:
                role = msg.get("role", "user")
                content = msg.get("content", "")
                if role == "system":
                    prompt_parts.append(f"System: {content}")
                elif role == "user":
                    prompt_parts.append(content)
                elif role == "assistant":
                    prompt_parts.append(f"Assistant: {content}")
            prompt = "\n\n".join(prompt_parts)
        
        if not prompt:
            return [{"text": ""}]  # dspy는 "text" 키를 기대함
        
        try:
            response = self.pool.generate_content(prompt)
            # dspy Adapter는 output["text"]를 기대함
            return [{"text": response}]
        except Exception as e:
            print(f"⚠️ Gemini Pool 에러: {e}")
            return [{"text": f"Error: {e}"}]

# Gemini Pool LM 인스턴스 생성
gemini_lm = GeminiPoolLM(gemini_pool, model_id="gemini-2.0-flash")

# dspy 기본 LM으로 설정
dspy.configure(lm=gemini_lm)

print(f"\n✅ dspy LM 설정 완료: Gemini Pool 사용")
print(f"   OPENAI_API_KEY 설정됨: {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")
print(f"   GOOGLE_API_KEY 설정됨: {'✅' if os.getenv('GOOGLE_API_KEY') else '❌ (polling_gemini 사용 중)'}")

/Users/jaehyeokchoi/Desktop/TableMagnifier/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-30 17:01:20,718 - polling_gemini.api_pool - INFO - 총 6개의 API 키를 로드했습니다.
2025-12-30 17:01:20,718 - polling_gemini.api_pool - INFO - API 키 'key1' 사용 중 (모델: gemini-2.5-flash)


✅ Gemini API Pool 초기화 완료
   - 사용 가능한 API 키: 6개
   - 현재 키: key1
   - 모델: models/gemini-2.5-flash

✅ dspy LM 설정 완료: Gemini Pool 사용
   OPENAI_API_KEY 설정됨: ❌
   GOOGLE_API_KEY 설정됨: ❌ (polling_gemini 사용 중)


### 5.1 QA Generation 프롬프트 최적화

In [17]:
# QA Generation 프롬프트 최적화 (polling_gemini 사용)
qa_optimizer = PydanticOptimizer(
    model=QAGenerationOutput,
    examples=qa_examples,
    lm=gemini_lm,  # polling_gemini LM 사용
    system_prompt="You are an expert in creating educational and reasoning questions from Korean tabular data.",
    instruction_prompt="Generate diverse Question-Answer pairs from the following HTML table. Include lookup, comparison, calculation, and reasoning type questions. All questions and answers should be in Korean.",
    verbose=True,
)

print("🚀 QA Generation 프롬프트 최적화 시작 (Gemini Pool 사용)...")
qa_result = qa_optimizer.optimize()

print("\n" + "="*60)
print("✅ QA Generation 최적화 완료!")
print("="*60)
print(f"\n📝 최적화된 Field Descriptions:")
for field, desc in qa_result.optimized_descriptions.items():
    print(f"  - {field}: {desc}")

if qa_result.optimized_system_prompt:
    print(f"\n📝 최적화된 System Prompt:")
    print(qa_result.optimized_system_prompt)

if qa_result.optimized_instruction_prompt:
    print(f"\n📝 최적화된 Instruction Prompt:")
    print(qa_result.optimized_instruction_prompt)

🚀 QA Generation 프롬프트 최적화 시작 (Gemini Pool 사용)...

Starting DSPy Pydantic optimization
Model: QAGenerationOutput
Optimizer: BOOTSTRAPFEWSHOT
Examples: 3
Fields to optimize: 4

Initial field descriptions (set during initialization):
  qa_pairs: 테이블에서 생성된 다양한 유형의 QA 쌍 목록
  qa_pairs.question: 테이블 데이터에 기반한 질문 (한국어)
  qa_pairs.answer: 질문에 대한 정확한 답변
  qa_pairs.type: 질문 유형: lookup(조회), comparison(비교), calculation(계산), reasoning(추론)
Optimization threads: 4

Training examples: 2
Validation examples: 1

Evaluating baseline configuration...
Baseline average score: 100.00%

Optimizing prompts and field descriptions...
  - System prompt
  - Instruction prompt
  - 4 field descriptions


 50%|█████     | 1/2 [01:03<01:03, 63.89s/it]2025-12-30 16:54:01,470 - polling_gemini.api_pool - WARNING - API 키 'key3' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 16:54:01,471 - polling_gemini.api_pool - INFO - API 키 'key1' 사용 중 (모델: gemini-2.5-flash)
2025-12-30 16:54:02,221 - polling_gemini.api_pool - WARNING - API 키 'key1' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 16:54:02,223 - polling_gemini.api_pool - INFO - API 키 'key2' 사용 중 (모델: gemini-2.5-flash)
100%|██████████| 2/2 [01:54<00:00, 57.34s/it]


Bootstrapped 2 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.


2025-12-30 16:54:47,048 - polling_gemini.api_pool - WARNING - API 키 'key2' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 16:54:47,051 - polling_gemini.api_pool - INFO - API 키 'key3' 사용 중 (모델: gemini-2.5-flash)



Evaluating optimized configuration...


2025-12-30 16:55:14,797 - polling_gemini.api_pool - WARNING - API 키 'key3' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 16:55:14,798 - polling_gemini.api_pool - INFO - API 키 'key3' 사용 중 (모델: gemini-2.5-flash)
2025-12-30 16:55:41,627 - polling_gemini.api_pool - WARNING - API 키 'key3' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 16:55:41,628 - polling_gemini.api_pool - INFO - API 키 'key3' 사용 중 (모델: gemini-2.5-flash)



Optimization complete
Baseline score: 100.00%
Final score: 100.00%
No change in performance.


✅ QA Generation 최적화 완료!

📝 최적화된 Field Descriptions:
  - qa_pairs: 테이블에서 생성된 다양한 유형의 QA 쌍
  - qa_pairs.question: Korean question about table data
  - qa_pairs.answer: 질문에 대한 정확하고 명확한 텍스트 답변
  - qa_pairs.type: 질문의 유형을 나타냅니다. 가능한 값은 'lookup', 'comparison', 'calculation', 'reasoning' 입니다.

📝 최적화된 System Prompt:
You are an expert at analyzing Korean tabular data to formulate high-quality educational and reasoning questions. Your questions should effectively assess comprehension, critical thinking, and logical inference, ensuring they are accurate, relevant, and directly derivable from the provided data.

📝 최적화된 Instruction Prompt:
From the provided HTML table, generate a diverse set of Question-Answer (Q&A) pairs. Each Q&A pair must be in Korean. Ensure a balanced representation of lookup, comparison, calculation, and reasoning question types. Present the output as a JSON array, where each object

### 5.2 Self-Reflection 프롬프트 최적화

In [ ]:
# Self-Reflection 프롬프트 최적화 (polling_gemini 사용)
reflection_optimizer = PydanticOptimizer(
    model=SelfReflectionOutput,
    examples=reflection_examples,
    lm=gemini_lm,  # polling_gemini LM 사용
    system_prompt="You are an expert evaluator for synthetic table generation quality.",
    instruction_prompt="Evaluate the quality of the generated synthetic table by comparing it with the original table. Check for data consistency, structural accuracy, and semantic preservation. Provide detailed feedback in Korean.",
    verbose=True,
)

print("🚀 Self-Reflection 프롬프트 최적화 시작 (Gemini Pool 사용)...")
reflection_result = reflection_optimizer.optimize()

print("\n" + "="*60)
print("✅ Self-Reflection 최적화 완료!")
print("="*60)
print(f"\n📝 최적화된 Field Descriptions:")
for field, desc in reflection_result.optimized_descriptions.items():
    print(f"  - {field}: {desc}")

🚀 Self-Reflection 프롬프트 최적화 시작 (Gemini Pool 사용)...

Starting DSPy Pydantic optimization
Model: SelfReflectionOutput
Optimizer: BOOTSTRAPFEWSHOT
Examples: 5
Fields to optimize: 6

Initial field descriptions (set during initialization):
  passed: 검증 통과 여부
  score: 품질 점수 (0-100)
  issues: 발견된 문제점 목록
  issues.type: 이슈 유형
  issues.detail: 이슈에 대한 상세 설명
  revision_instructions: 테이블 수정을 위한 구체적인 단계별 지시사항
Optimization threads: 4

Training examples: 4
Validation examples: 1

Evaluating baseline configuration...


/Users/jaehyeokchoi/Desktop/TableMagnifier/.venv/lib/python3.12/site-packages/dspydantic/optimizer.py:821: UserWarning: No template placeholders found in instruction prompt, but text_dict was provided. Appending values: original_html: <table>
            <tr><th>이름</th><th>부서</th><th>연봉</th></tr>
            <tr><td>김철수</td><td>개발팀</td><td>5,000만원</td></tr>
            <tr><td>이영희</td><td>마케팅</td><td>4,500만원</td></tr>
            </table>, synthetic_html: <table>
            <tr><th>이름</th><th>부서</th><th>연봉</th></tr>
            <tr><td>박민수</td><td>인사팀</td><td>4,800만원</td></tr>
            <tr><td>최지은</td><td>기획팀</td><td>5,200만원</td></tr>
            </table>
  formatted_instruction = format_instruction_prompt_template(
/Users/jaehyeokchoi/Desktop/TableMagnifier/.venv/lib/python3.12/site-packages/dspydantic/optimizer.py:821: UserWarning: No template placeholders found in instruction prompt, but text_dict was provided. Appending values: original_html: <table>
            <tr><th>제품</th>

Baseline average score: 60.00%

Optimizing prompts and field descriptions...
  - System prompt
  - Instruction prompt
  - 6 field descriptions


  0%|          | 0/4 [00:00<?, ?it/s]2025-12-30 17:01:53,690 - polling_gemini.api_pool - WARNING - API 키 'key1' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 17:01:53,691 - polling_gemini.api_pool - INFO - API 키 'key2' 사용 중 (모델: gemini-2.5-flash)
 25%|██▌       | 1/4 [01:00<03:02, 60.84s/it]2025-12-30 17:02:53,584 - polling_gemini.api_pool - WARNING - API 키 'key2' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 17:02:53,586 - polling_gemini.api_pool - INFO - API 키 'key3' 사용 중 (모델: gemini-2.5-flash)
 50%|█████     | 2/4 [01:54<01:53, 56.70s/it]2025-12-30 17:03:27,861 - polling_gemini.api_pool - WARNING - API 키 'key3' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 17:03:27,863 - polling_gemini.api_pool - INFO - API 키 'key4' 사용 중 (모델: gemini-2.5-flash)
2025-12-30 17:04:17,115 - polling_gemini.api_pool - WARNING - API 키 'key4' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 17:04:17,117 - polling_gemini.api_pool - INFO - API 키 'key5' 사용 중 (모델: gemini-2.5-flash)
 75%|███████▌  | 3/4 [03:02<01:01, 62.00s/it]2025-12-30 17:05:09,389 - polling_gemini.api

Bootstrapped 4 full traces after 3 examples for up to 1 rounds, amounting to 4 attempts.


2025-12-30 17:05:45,693 - polling_gemini.api_pool - WARNING - API 키 'key6' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 17:05:45,695 - polling_gemini.api_pool - INFO - API 키 'key1' 사용 중 (모델: gemini-2.5-flash)



Evaluating optimized configuration...


2025-12-30 17:06:14,571 - polling_gemini.api_pool - WARNING - API 키 'key1' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 17:06:14,573 - polling_gemini.api_pool - INFO - API 키 'key2' 사용 중 (모델: gemini-2.5-flash)
2025-12-30 17:06:39,661 - polling_gemini.api_pool - WARNING - API 키 'key2' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 17:06:39,662 - polling_gemini.api_pool - INFO - API 키 'key3' 사용 중 (모델: gemini-2.5-flash)



Optimization complete
Baseline score: 60.00%
Final score: 60.00%
No change in performance.


✅ Self-Reflection 최적화 완료!

📝 최적화된 Field Descriptions:
  - passed: 검증 통과
  - score: 품질 점수 (0-100 범위의 정수)
  - issues: 회고 문제점
  - issues.type: 이슈 유형
  - issues.detail: 이슈 상세 설명
  - revision_instructions: 테이블 수정 단계별 지시사항


In [12]:
if reflection_result.optimized_system_prompt:
    print(f"\n📝 최적화된 System Prompt:")
    print(reflection_result.optimized_system_prompt)

if reflection_result.optimized_instruction_prompt:
    print(f"\n📝 최적화된 Instruction Prompt:")
    print(reflection_result.optimized_instruction_prompt)


📝 최적화된 System Prompt:
You are a highly skilled expert evaluator specializing in the quality assessment of synthetic table generation. Your primary task is to critically analyze and provide comprehensive, actionable feedback on the fidelity, realism, statistical integrity, and overall utility of generated synthetic tables.

📝 최적화된 Instruction Prompt:
Given an `original_html` table and a `synthetic_html` table, your task is to evaluate the quality of the `synthetic_html` by comparing it against the `original_html`.

Your evaluation should focus on the following aspects:
1.  **Structural Accuracy**: Assess if the synthetic table maintains the same number of rows, columns, and overall HTML table structure as the original.
2.  **Data Consistency**: Check for consistency in data types, formats, and values between corresponding cells.
3.  **Semantic Preservation**: Determine if the meaning, context, and relationships of the data and columns are preserved.

Provide a detailed feedback report 

### 5.3 Table Validation 프롬프트 최적화

In [9]:
# Table Validation 프롬프트 최적화 (polling_gemini 사용)
validation_optimizer = PydanticOptimizer(
    model=TableValidationOutput,
    examples=validation_examples,
    lm=gemini_lm,  # polling_gemini LM 사용
    system_prompt="You are an expert at validating data consistency between tables.",
    instruction_prompt="Compare the original table with the generated synthetic table. Validate structural consistency, value accuracy, and completeness. Provide detailed results in Korean.",
    verbose=True,
)

print("🚀 Table Validation 프롬프트 최적화 시작 (Gemini Pool 사용)...")
validation_result = validation_optimizer.optimize()

print("\n" + "="*60)
print("✅ Table Validation 최적화 완료!")
print("="*60)
print(f"\n📝 최적화된 Field Descriptions:")
for field, desc in validation_result.optimized_descriptions.items():
    print(f"  - {field}: {desc}")

🚀 Table Validation 프롬프트 최적화 시작 (Gemini Pool 사용)...

Starting DSPy Pydantic optimization
Model: TableValidationOutput
Optimizer: BOOTSTRAPFEWSHOT
Examples: 5
Fields to optimize: 2

Initial field descriptions (set during initialization):
  valid: 테이블 유효성 여부
  reason: 판단 이유에 대한 간단한 설명
Optimization threads: 4

Training examples: 4
Validation examples: 1

Evaluating baseline configuration...
Baseline average score: 50.00%

Optimizing prompts and field descriptions...
  - System prompt
  - Instruction prompt
  - 2 field descriptions


 25%|██▌       | 1/4 [00:29<01:27, 29.10s/it]2025-12-30 17:10:45,145 - polling_gemini.api_pool - WARNING - API 키 'key3' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 17:10:45,147 - polling_gemini.api_pool - INFO - API 키 'key4' 사용 중 (모델: gemini-2.5-flash)
 75%|███████▌  | 3/4 [01:37<00:32, 32.42s/it]2025-12-30 17:12:03,424 - polling_gemini.api_pool - WARNING - API 키 'key4' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 17:12:03,426 - polling_gemini.api_pool - INFO - API 키 'key5' 사용 중 (모델: gemini-2.5-flash)
100%|██████████| 4/4 [02:07<00:00, 31.81s/it]


Bootstrapped 2 full traces after 3 examples for up to 1 rounds, amounting to 4 attempts.

Evaluating optimized configuration...


2025-12-30 17:12:46,836 - polling_gemini.api_pool - WARNING - API 키 'key5' 할당량 초과. 다음 키로 전환합니다.
2025-12-30 17:12:46,837 - polling_gemini.api_pool - INFO - API 키 'key6' 사용 중 (모델: gemini-2.5-flash)



Optimization complete
Baseline score: 50.00%
Final score: 50.00%
No change in performance.


✅ Table Validation 최적화 완료!

📝 최적화된 Field Descriptions:
  - valid: 테이블 유효 여부
  - reason: 판단 근거


In [13]:
if validation_result.optimized_system_prompt:
    print(f"\n📝 최적화된 System Prompt:")
    print(validation_result.optimized_system_prompt)

if validation_result.optimized_instruction_prompt:
    print(f"\n📝 최적화된 Instruction Prompt:")
    print(validation_result.optimized_instruction_prompt)


📝 최적화된 System Prompt:
You are a highly skilled Data Consistency Validator. Your core function is to meticulously identify, analyze, and report discrepancies, anomalies, and inconsistencies across various datasets and relational database tables. Your objective is to ensure the highest level of data integrity, accuracy, and reliability.

📝 최적화된 Instruction Prompt:
Conduct a comprehensive evaluation of the generated synthetic table against the original table.
Specifically, validate the following aspects:
1.  **Structural Consistency:** Verify column names, data types, and table dimensions (number of rows and columns).
2.  **Value Accuracy:** Assess the similarity in data distributions, value ranges, and identify any significant discrepancies in specific values.
3.  **Completeness:** Evaluate the presence of missing values and overall data coverage.

Generate a detailed validation report in Korean, including a clear summary of findings, identified discrepancies, and an overall assessment 

## 6. 최적화 결과 저장

In [ ]:
import json
from datetime import datetime

# 결과 저장
optimization_results = {
    "timestamp": datetime.now().isoformat(),
    "model_id": MODEL_ID,
    "qa_generation": {
        "optimized_descriptions": qa_result.optimized_descriptions,
        "optimized_system_prompt": qa_result.optimized_system_prompt,
        "optimized_instruction_prompt": qa_result.optimized_instruction_prompt,
    },
    "self_reflection": {
        "optimized_descriptions": reflection_result.optimized_descriptions,
        "optimized_system_prompt": reflection_result.optimized_system_prompt,
        "optimized_instruction_prompt": reflection_result.optimized_instruction_prompt,
    },
    "table_validation": {
        "optimized_descriptions": validation_result.optimized_descriptions,
        "optimized_system_prompt": validation_result.optimized_system_prompt,
        "optimized_instruction_prompt": validation_result.optimized_instruction_prompt,
    },
}

# JSON으로 저장
output_path = Path.cwd() / "optimization_results.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(optimization_results, f, ensure_ascii=False, indent=2)

print(f"✅ 최적화 결과 저장됨: {output_path}")

## 7. 최적화된 프롬프트를 YAML로 내보내기

In [ ]:
def generate_optimized_prompt(original_prompt: str, result) -> str:
    """
    최적화 결과를 바탕으로 새로운 프롬프트 생성.
    system_prompt와 instruction_prompt를 결합.
    """
    parts = []
    
    if result.optimized_system_prompt:
        parts.append(f"[System Context]\n{result.optimized_system_prompt}")
    
    if result.optimized_instruction_prompt:
        parts.append(f"[Instruction]\n{result.optimized_instruction_prompt}")
    
    if result.optimized_descriptions:
        desc_text = "\n".join([f"  - {k}: {v}" for k, v in result.optimized_descriptions.items()])
        parts.append(f"[Expected Output Fields]\n{desc_text}")
    
    # 원본 프롬프트의 Output Format 부분 유지
    if "**Output Format" in original_prompt:
        format_start = original_prompt.find("**Output Format")
        parts.append(original_prompt[format_start:])
    
    return "\n\n".join(parts)

# 최적화된 YAML 생성
optimized_prompts = prompts.copy()

# QA Generation 업데이트
if 'generate_qa' in prompts:
    optimized_prompts['generate_qa'] = generate_optimized_prompt(
        prompts['generate_qa'], qa_result
    )

# Self-Reflection 업데이트  
if 'self_reflection' in prompts:
    optimized_prompts['self_reflection'] = generate_optimized_prompt(
        prompts['self_reflection'], reflection_result
    )

# Validation 업데이트
if 'validate_parsed_table' in prompts:
    optimized_prompts['validate_parsed_table'] = generate_optimized_prompt(
        prompts['validate_parsed_table'], validation_result
    )

# 최적화된 YAML 저장
optimized_yaml_path = PROMPTS_DIR / "optimized.yaml"
with open(optimized_yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(optimized_prompts, f, allow_unicode=True, default_flow_style=False, sort_keys=False)

print(f"✅ 최적화된 프롬프트 저장됨: {optimized_yaml_path}")

## 8. 최적화 전후 비교

In [ ]:
print("="*60)
print("📊 QA Generation 프롬프트 비교")
print("="*60)

print("\n[원본 System Prompt 개념]")
print("You are an expert in creating educational and reasoning questions from tabular data.")

print("\n[최적화된 System Prompt]")
print(qa_result.optimized_system_prompt or "(변경 없음)")

print("\n" + "-"*60)

print("\n[원본 Field Descriptions]")
print("  - question: 테이블 데이터에 기반한 질문")
print("  - answer: 질문에 대한 답변")
print("  - type: lookup/comparison/calculation/reasoning")

print("\n[최적화된 Field Descriptions]")
for field, desc in qa_result.optimized_descriptions.items():
    print(f"  - {field}: {desc}")

## 9. 이미지 기반 프롬프트 최적화 (선택사항)

`generate_qa_from_image`, `generate_synthetic_table_from_image` 등 이미지 입력이 필요한 프롬프트는 실제 이미지 예제와 함께 최적화할 수 있습니다.

In [ ]:
# 이미지 기반 예제 (sample_images 폴더의 이미지 사용)
SAMPLE_IMAGES_DIR = PROJECT_ROOT / "tests" / "choi" / "Table_example" / "sample_images"

# 이미지 파일 목록
image_files = list(SAMPLE_IMAGES_DIR.glob("*.png"))
print(f"사용 가능한 샘플 이미지: {len(image_files)}개")
for img in image_files:
    print(f"  - {img.name}")

In [ ]:
# 이미지 기반 QA 생성 최적화 예제 (이미지가 있는 경우)
if image_files:
    image_qa_examples = [
        Example(
            image_path=str(image_files[0]),
            expected_output=QAGenerationOutput(
                qa_pairs=[
                    # 이미지에 맞는 실제 QA 쌍으로 교체 필요
                    QAPair(question="XX+3세의 기준보험료는?", answer="89,030", type="lookup"),
                    QAPair(question="나이증가분이 가장 높은 연령대는?", answer="XX+5세", type="comparison"),
                ]
            )
        )
    ]
    
    print(f"이미지 기반 예제 준비됨: {len(image_qa_examples)}개")
    print("\n⚠️ 이미지 기반 최적화를 실행하려면 expected_output을 실제 이미지 내용에 맞게 수정하세요.")
else:
    print("⚠️ 샘플 이미지가 없습니다.")